In [1]:
# %pip install google


In [2]:
# !pip install ultralytics

In [3]:
pip install deep-sort-realtime

Note: you may need to restart the kernel to use updated packages.


In [4]:
# import torch
# import torch.nn as nn
# import numpy as np
# import cv2
# from ultralytics import YOLO
# from deep_sort_realtime.deepsort_tracker import DeepSort
# from google.colab.patches import cv2_imshow
# # ==============================
# # 1. DEVICE SETUP
# # ==============================
# device = torch.device("cpu")
# print("Using device:", device)

# # ==============================
# # 2. GRU MODEL
# # ==============================
# class TrajectoryGRU(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.gru = nn.GRU(2, 64, 2, batch_first=True)
#         self.fc = nn.Linear(64, 40)

#     def forward(self, x):
#         out, _ = self.gru(x)
#         out = out[:, -1, :]
#         out = self.fc(out)
#         return out.view(-1, 20, 2)

# # ==============================
# # 3. LOAD MODEL
# # ==============================
# model = TrajectoryGRU().to(device)
# try:
#     # Ensure this path is correct for your local machine
#     model.load_state_dict(torch.load(
#         "/content/gru_model.pth",
#         map_location=device
#     ))
#     model.eval()
#     print("GRU model loaded successfully")
# except Exception as e:
#     print(f"Error loading model: {e}")
#     exit()

# # ==============================
# # 4. YOLO + TRACKER
# # ==============================
# yolo = YOLO("yolov8n.pt")
# tracker = DeepSort(max_age=30)

# # ==============================
# # 5. VIDEO & CONSTANTS
# # ==============================
# video_path = "/Users/krishnanshuramjiwala/Downloads/Projects/AAIDL Project/video/Pedestrian_Trajectory_Demo_Video_Generated.mp4"
# cap = cv2.VideoCapture(video_path)

# if not cap.isOpened():
#     print(f"Error: Video not found at {video_path}")
#     exit()

# INPUT_SEQ_TIME = 30
# MODEL_INPUT_SEQ = 50
# X_TRAIN_MIN, X_TRAIN_MAX = 30.0, 746.0
# Y_TRAIN_MIN, Y_TRAIN_MAX = 74.0, 744.0

# trajectories = {}

# # ==============================
# # 6. MAIN LOOP
# # ==============================
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     h, w, _ = frame.shape
#     results = yolo(frame, verbose=False)[0]
#     detections = []

#     for box in results.boxes:
#         if int(box.cls[0]) == 0:
#             x1, y1, x2, y2 = box.xyxy[0]
#             conf = float(box.conf[0])
#             detections.append(([float(x1), float(y1), float(x2-x1), float(y2-y1)], conf, 'person'))

#     tracks = tracker.update_tracks(detections, frame=frame)

#     for track in tracks:
#         if not track.is_confirmed():
#             continue

#         track_id = track.track_id
#         l, t, r, b = track.to_ltrb()
#         #cx, cy = (l + r) / 2, (t + b) / 2
#         # Anchor the trajectory to the bottom-center (the feet) instead of the torso
#         cx, cy = (l + r) / 2, b

#         if track_id not in trajectories:
#             trajectories[track_id] = []

#         trajectories[track_id].append([cx, cy])
#         if len(trajectories[track_id]) > INPUT_SEQ_TIME:
#             trajectories[track_id].pop(0)

#         traj = trajectories[track_id]

#         # --- DRAW PAST PATH (BLUE) ---
#         for i in range(1, len(traj)):
#             cv2.line(frame, (int(traj[i-1][0]), int(traj[i-1][1])),
#                      (int(traj[i][0]), int(traj[i][1])), (255, 0, 0), 2)

#         # --- PREDICTION LOGIC ---
#        # --- PREDICTION LOGIC ---
#         if len(traj) >= 15:
#             traj_np = np.array(traj, dtype=np.float32)

#             # 1. Map to Dataset Coordinates (The scale the model expects)
#             input_seq = traj_np.copy()
#             input_seq[:, 0] = np.interp(input_seq[:, 0], [0, w], [X_TRAIN_MIN, X_TRAIN_MAX])
#             input_seq[:, 1] = np.interp(input_seq[:, 1], [0, h], [Y_TRAIN_MIN, Y_TRAIN_MAX])

#             # 2. Resample to 50 points
#             old_idx = np.linspace(0, 1, len(input_seq))
#             new_idx = np.linspace(0, 1, MODEL_INPUT_SEQ)
#             input_seq = np.stack([
#                 np.interp(new_idx, old_idx, input_seq[:, 0]),
#                 np.interp(new_idx, old_idx, input_seq[:, 1])
#             ], axis=1)

#             # 3. Center the sequence
#             origin = input_seq[-1].copy()
#             input_seq_centered = input_seq - origin

#             input_tensor = torch.tensor(input_seq_centered, dtype=torch.float32).unsqueeze(0).to(device)

#             with torch.no_grad():
#                 pred = model(input_tensor).cpu().numpy()[0]

#             # --- VISUALIZATION & AMPLIFICATION ---
#             first_tracked_id = list(trajectories.keys())[0]
#             if track_id == first_tracked_id:

#                 # Accumulate the step-by-step velocities into a path from (0,0)
#                 pred_path_dataset = np.cumsum(pred, axis=0)

#                 # Calculate how to scale dataset units back to screen pixels
#                 scale_x = w / (X_TRAIN_MAX - X_TRAIN_MIN)
#                 scale_y = h / (Y_TRAIN_MAX - Y_TRAIN_MIN)

#                 # 🚀 AMPLIFIER: Increase this if the line is still too short, decrease if too long!
#                 AMPLIFIER = 8.0

#                 pred_pixels = []
#                 last_pixel_x, last_pixel_y = traj[-1][0], traj[-1][1]

#                 for dx_ds, dy_ds in pred_path_dataset:
#                     # Convert dataset movement to pixel movement and amplify it
#                     dx_px = dx_ds * scale_x * AMPLIFIER
#                     dy_px = dy_ds * scale_y * AMPLIFIER

#                     # Add the amplified movement to the pedestrian's actual hip location
#                     px = int(last_pixel_x + dx_px)
#                     py = int(last_pixel_y + dy_px)
#                     pred_pixels.append((px, py))

#                 # Draw the path
#                 short_prediction = pred_pixels[:10] # Show 10 steps into the future
#                 if len(short_prediction) > 1:
#                     for i in range(len(short_prediction) - 1):
#                         cv2.line(frame, short_prediction[i], short_prediction[i+1], (0, 0, 255), 3)
#                     for pt in short_prediction[::2]:
#                         cv2.circle(frame, pt, 4, (0, 0, 255), -1)

#         # --- DRAW BOUNDING BOX ---

#         # --- DRAW BOUNDING BOX ---
#         cv2.rectangle(frame, (int(l), int(t)), (int(r), int(b)), (0, 255, 0), 2)
#         cv2.putText(frame, f"ID {track_id}", (int(l), int(t)-10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

#     cv2_imshow(frame)
#     if cv2.waitKey(1) == 27:
#         break

# cap.release()
# cv2.destroyAllWindows()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ==============================
# 1. DEVICE SETUP
# ==============================
device = torch.device("cpu")
print("Using device:", device)

# ==============================
# 2. GRU MODEL
# ==============================
class TrajectoryGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(2, 64, 2, batch_first=True)
        self.fc = nn.Linear(64, 40)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.view(-1, 20, 2)

# ==============================
# 3. LOAD MODEL
# ==============================
model = TrajectoryGRU().to(device)

try:
    model.load_state_dict(torch.load(
        "/Users/krishnanshuramjiwala/Downloads/gru_model.pth",  # ✅ FIXED PATH
        map_location=device
    ))
    model.eval()
    print("GRU model loaded successfully")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# ==============================
# 4. YOLO + TRACKER
# ==============================
yolo = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30)

# ==============================
# 5. VIDEO
# ==============================
video_path = "/Users/krishnanshuramjiwala/Downloads/Projects/AAIDL Project/video/Pedestrian_Trajectory_Demo_Video_Generated.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Video not found at {video_path}")
    exit()

INPUT_SEQ_TIME = 30
MODEL_INPUT_SEQ = 50
X_TRAIN_MIN, X_TRAIN_MAX = 30.0, 746.0
Y_TRAIN_MIN, Y_TRAIN_MAX = 74.0, 744.0

trajectories = {}

# ==============================
# 6. MAIN LOOP
# ==============================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    results = yolo(frame, verbose=False)[0]
    detections = []

    for box in results.boxes:
        if int(box.cls[0]) == 0:
            x1, y1, x2, y2 = box.xyxy[0]
            conf = float(box.conf[0])
            detections.append(([float(x1), float(y1), float(x2-x1), float(y2-y1)], conf, 'person'))

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = track.to_ltrb()
        cx, cy = (l + r) / 2, b

        if track_id not in trajectories:
            trajectories[track_id] = []

        trajectories[track_id].append([cx, cy])
        if len(trajectories[track_id]) > INPUT_SEQ_TIME:
            trajectories[track_id].pop(0)

        traj = trajectories[track_id]

        # Draw past path
        for i in range(1, len(traj)):
            cv2.line(frame, (int(traj[i-1][0]), int(traj[i-1][1])),
                     (int(traj[i][0]), int(traj[i][1])), (255, 0, 0), 2)

        # Prediction
        if len(traj) >= 15:
            traj_np = np.array(traj, dtype=np.float32)

            input_seq = traj_np.copy()
            input_seq[:, 0] = np.interp(input_seq[:, 0], [0, w], [X_TRAIN_MIN, X_TRAIN_MAX])
            input_seq[:, 1] = np.interp(input_seq[:, 1], [0, h], [Y_TRAIN_MIN, Y_TRAIN_MAX])

            old_idx = np.linspace(0, 1, len(input_seq))
            new_idx = np.linspace(0, 1, MODEL_INPUT_SEQ)

            input_seq = np.stack([
                np.interp(new_idx, old_idx, input_seq[:, 0]),
                np.interp(new_idx, old_idx, input_seq[:, 1])
            ], axis=1)

            origin = input_seq[-1].copy()
            input_seq_centered = input_seq - origin

            input_tensor = torch.tensor(input_seq_centered, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                pred = model(input_tensor).cpu().numpy()[0]

            pred_path_dataset = np.cumsum(pred, axis=0)

            scale_x = w / (X_TRAIN_MAX - X_TRAIN_MIN)
            scale_y = h / (Y_TRAIN_MAX - Y_TRAIN_MIN)

            AMPLIFIER = 8.0

            pred_pixels = []
            last_pixel_x, last_pixel_y = traj[-1][0], traj[-1][1]

            for dx_ds, dy_ds in pred_path_dataset:
                dx_px = dx_ds * scale_x * AMPLIFIER
                dy_px = dy_ds * scale_y * AMPLIFIER

                px = int(last_pixel_x + dx_px)
                py = int(last_pixel_y + dy_px)
                pred_pixels.append((px, py))

            short_prediction = pred_pixels[:10]
            for i in range(len(short_prediction) - 1):
                cv2.line(frame, short_prediction[i], short_prediction[i+1], (0, 0, 255), 3)

        cv2.rectangle(frame, (int(l), int(t)), (int(r), int(b)), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {track_id}", (int(l), int(t)-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # ✅ FIXED DISPLAY
    cv2.imshow("Output", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC key
        break

cap.release()
cv2.destroyAllWindows()

Using device: cpu
Error loading model: [Errno 2] No such file or directory: '/Users/krishnanshuramjiwala/Downloads/gru_model.pth'


/opt/miniconda3/envs/traffic_env/lib/python3.10/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


: 